In [50]:
print("hello from llmops")

hello from llmops


In [51]:
from dotenv import load_dotenv
load_dotenv()

True

## Creating the dataset 

In [34]:
import pandas as pd

inputs = [
    "Who was Adolf Hitler and why is he considered one of the most infamous figures in history?",
    "Where and when was Adolf Hitler born?",
    "What career did Hitler initially aspire to before entering politics?",
    "Which event marked Hitler’s first major failed attempt to seize power?",
    "What book did Hitler write during his imprisonment and what ideology did it promote?",
    "In which year was Adolf Hitler appointed Chancellor of Germany?",
    "What laws stripped Jews of their citizenship and civil rights in Nazi Germany?",
    "Which invasion by Nazi Germany led Britain and France to declare war?",
    "What military tactic did Nazi Germany use to rapidly conquer much of Europe?",
    "What were two major strategic mistakes made by Hitler during World War II?"
]

outputs = [
    "Adolf Hitler was a German politician and dictator who led the Nazi Party and is infamous for initiating World War II and orchestrating the Holocaust, which caused millions of deaths.",
    "Adolf Hitler was born on April 20, 1889, in Braunau am Inn, Austria-Hungary, which is now modern-day Austria.",
    "Hitler initially aspired to become an artist but was rejected twice by the Academy of Fine Arts Vienna.",
    "Hitler’s first major failed attempt to seize power was the Beer Hall Putsch in Munich in 1923.",
    "Hitler wrote Mein Kampf during his imprisonment, which promoted anti-Semitism, Aryan supremacy, anti-communism, and the idea of Lebensraum for Germans.",
    "Adolf Hitler was appointed Chancellor of Germany in 1933.",
    "The Nuremberg Laws of 1935 stripped Jews of their citizenship and civil rights in Nazi Germany.",
    "The invasion of Poland in 1939 by Nazi Germany led Britain and France to declare war.",
    "Nazi Germany used the Blitzkrieg tactic to rapidly conquer much of Europe.",
    "Two major strategic mistakes by Hitler were invading the Soviet Union and declaring war on the United States."
]

qa_pairs = [{"question":q , "answer" : a} for q, a in zip(inputs , outputs)]
df = pd.DataFrame(qa_pairs)


csv_path = "C:/Users/PC/OneDrive/Desktop/llm_ops/data/QaPairs.csv"
df.to_csv(csv_path, index=False)

## Create the dataset in Langsmith 

In [52]:
# we have to make the client for that 

from langsmith import Client

client = Client()
dataset_name = "hitler_llmops_dataset_2"

# creating the dataset in langsmith
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Input and expected output pairs for Adolf Hitler",
)

# Storing question–answer examples in the dataset
client.create_examples(
    inputs=[{"question": q} for q in inputs],
    outputs=[{"answer": a} for a in outputs],
    dataset_id=dataset.id,
)

{'example_ids': ['9bf58de9-0114-4bd0-a1d9-9141485753df',
  'a0cc0776-617e-4337-bb0b-e6c1519fa1bf',
  '6408fb93-9f60-400a-ba0e-25c3d1a1dc46',
  '83b01f09-7321-4922-ab7a-31221d79ca3a',
  '141d8ce0-321f-4b63-83ca-7e2d506008f6',
  '9e9dd8d0-81a1-4b4e-a522-69f8c035da2e',
  '81e726cd-7633-478f-8de5-1dab62ee8083',
  'f6804d7f-caf6-4e08-a1f7-22221fa1e470',
  'fc243fb2-e362-4d49-b7d7-24e6f42087b8',
  '6af1dce7-b5bf-4b7c-aa44-7d173bed1e08'],
 'count': 10}

In [53]:
import sys
sys.path.append("C:/Users/PC/OneDrive/Desktop/llm_ops")

from pathlib import Path
from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG
import os

# Simple file adapter for local file paths
class LocalFileAdapter: # This adapter pretends a local file is an uploaded file
    """Adapter for local file paths to work with ChatIngestor."""
    def __init__(self, file_path: str):
        self.path = Path(file_path)
        self.name = self.path.name
    
    def getbuffer(self) -> bytes: # return file content in raw bytes beacus the ingestion pipelines usually read binary data
        return self.path.read_bytes()


def answer_ai_report_question(
    inputs: dict,
    data_path: str = "C:/Users/PC/OneDrive/Desktop/llm_ops/data/Hitler.txt",
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    k: int = 5
) -> dict:
    """
    Answer questions about the Adolf Hitler Report using RAG.
    
    Args:
        inputs: Dictionary containing the question, e.g., {"question": "Who was Adolf Hitler?"}
        data_path: Path to the AI Engineering Report text file
        chunk_size: Size of text chunks for splitting
        chunk_overlap: Overlap between chunks
        k: Number of documents to retrieve
    
    Returns:
        Dictionary with the answer, e.g., {"answer": "RAG stands for..."}
    """
    try:
        # Extract question from inputs
        question = inputs.get("question", "")
        if not question:
            return {"answer": "No question provided"}
        
        # Check if file exists
        if not Path(data_path).exists():
            return {"answer": f"Data file not found: {data_path}"}
        
        # Create file adapter
        file_adapter = LocalFileAdapter(data_path)
        
        # Build index using ChatIngestor
        ingestor = ChatIngestor(
            temp_base="data",
            faiss_base="faiss_index",
            use_session_dirs=True
        )
        
        # Build retriever
        ingestor.built_retriver(
            uploaded_files=[file_adapter],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            k=k
        )
        
        # Get session ID and index path
        session_id = ingestor.session_id
        index_path = f"faiss_index/{session_id}"
        
        # Create RAG instance and load retriever
        rag = ConversationalRAG(session_id=session_id)
        rag.load_retriever_from_faiss(
            index_path=index_path,
            k=k,
            index_name=os.getenv("FAISS_INDEX_NAME", "index")
        )
        
        # Get answer
        answer = rag.invoke(question, chat_history=[])
        
        return {"answer": answer}
        
    except Exception as e:
        return {"answer": f"Error: {str(e)}"}

In [54]:
print("Testing all questions from the dataset:\n")
for i, q in enumerate(inputs, 1):
    test_input = {"question": q}
    result = answer_ai_report_question(test_input)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}\n")
    print("-" * 80 + "\n")

{"timestamp": "2025-12-16T10:32:31.184757Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2025-12-16T10:32:31.185305Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2025-12-16T10:32:31.185724Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_6r...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2025-12-16T10:32:31.186348Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2025-12-16T10:32:31.188981Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20251216_153231_def00c53", "temp_dir": "data\\session_20251216_153231_def00c53", "faiss_dir": "faiss_index\\session_20251216_153231_def00c53", "sessionized": true, "timestamp": "2025-12-16T10:32:31.190348Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "Hitler.txt", "saved_as": "data\\session_2025

Testing all questions from the dataset:



{"added": 1, "index": "faiss_index\\session_20251216_153231_def00c53", "timestamp": "2025-12-16T10:32:33.421574Z", "level": "info", "event": "FAISS index updated"}
{"k": 5, "fetch_k": 20, "lambda_mult": 0.5, "timestamp": "2025-12-16T10:32:33.422264Z", "level": "info", "event": "Using MMR search"}
{"timestamp": "2025-12-16T10:32:33.424276Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2025-12-16T10:32:33.424765Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2025-12-16T10:32:33.425272Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_6r...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2025-12-16T10:32:33.425714Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2025-12-16T10:32:33.427476Z", "level": "info", "event": "YAML config loaded"}
{"provider": "groq", "model": "llama-3.1-8b-in

Q1: Who was Adolf Hitler and why is he considered one of the most infamous figures in history?
A1: Adolf Hitler was a German politician and dictator who led the National Socialist German Workers' Party (Nazi Party). He is considered one of the most infamous figures in history due to his central role in initiating World War II and orchestrating the Holocaust, which led to the deaths of millions.

--------------------------------------------------------------------------------



{"added": 1, "index": "faiss_index\\session_20251216_153237_a3d9091e", "timestamp": "2025-12-16T10:32:40.092189Z", "level": "info", "event": "FAISS index updated"}
{"k": 5, "fetch_k": 20, "lambda_mult": 0.5, "timestamp": "2025-12-16T10:32:40.092854Z", "level": "info", "event": "Using MMR search"}
{"timestamp": "2025-12-16T10:32:40.094814Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2025-12-16T10:32:40.095348Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2025-12-16T10:32:40.095818Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_6r...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2025-12-16T10:32:40.096276Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2025-12-16T10:32:40.097970Z", "level": "info", "event": "YAML config loaded"}
{"provider": "groq", "model": "llama-3.1-8b-in

...

Q8: Which invasion by Nazi Germany led Britain and France to declare war?
A8: The invasion by Nazi Germany that led Britain and France to declare war was the invasion of Poland in 1939.

--------------------------------------------------------------------------------



{"added": 1, "index": "faiss_index\\session_20251216_153311_9230b9d3", "timestamp": "2025-12-16T10:33:14.604301Z", "level": "info", "event": "FAISS index updated"}
{"k": 5, "fetch_k": 20, "lambda_mult": 0.5, "timestamp": "2025-12-16T10:33:14.605005Z", "level": "info", "event": "Using MMR search"}
{"timestamp": "2025-12-16T10:33:14.607309Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2025-12-16T10:33:14.607916Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2025-12-16T10:33:14.608854Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_6r...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2025-12-16T10:33:14.609475Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2025-12-16T10:33:14.611301Z", "level": "info", "event": "YAML config loaded"}
{"provider": "groq", "model": "llama-3.1-8b-in

Q9: What military tactic did Nazi Germany use to rapidly conquer much of Europe?
A9: Nazi Germany used the **Blitzkrieg** tactic to rapidly conquer much of Europe.

--------------------------------------------------------------------------------



{"added": 1, "index": "faiss_index\\session_20251216_153320_d870820d", "timestamp": "2025-12-16T10:33:22.792633Z", "level": "info", "event": "FAISS index updated"}
{"k": 5, "fetch_k": 20, "lambda_mult": 0.5, "timestamp": "2025-12-16T10:33:22.793387Z", "level": "info", "event": "Using MMR search"}
{"timestamp": "2025-12-16T10:33:22.795376Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2025-12-16T10:33:22.795907Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2025-12-16T10:33:22.796393Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_6r...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2025-12-16T10:33:22.796850Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2025-12-16T10:33:22.798476Z", "level": "info", "event": "YAML config loaded"}
{"provider": "groq", "model": "llama-3.1-8b-in

Q10: What were two major strategic mistakes made by Hitler during World War II?
A10: Two major strategic mistakes made by Hitler during World War II were:

1. **Invading the Soviet Union (Operation Barbarossa in 1941)**: This invasion stretched German resources, resulted in massive losses, especially in the **Battle of Stalingrad**, and ultimately led to the Soviet Union's counterattack that pushed the Germans back.
2. **Declaring war on the United States after Pearl Harbor**: This decision brought the industrial power of America into the conflict, which significantly shifted the balance of power against the Axis Powers.

--------------------------------------------------------------------------------



## Performing the langsmith evaluation

In [38]:
from langsmith.evaluation import evaluate, LangChainStringEvaluator

In [58]:
from langchain_groq import ChatGroq
eval_llm = ChatGroq(
    model="llama-3.3-70b-versatile", 
    temperature=0
)

In [59]:
eval_llm = ChatGroq(
    model="llama-3.3-70b-versatile", 
    temperature=0
)

In [60]:
qa_evaluator = [
    LangChainStringEvaluator(
        "cot_qa", 
        config={"llm": eval_llm}
    )
]

In [62]:

dataset_name = "hitler_llmops_dataset_2"

# Run evaluation using our RAG function
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=qa_evaluator,
    experiment_prefix="test-hitler_llmops_dataset_2-qa-rag",
    # Experiment metadata
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "evaluator_model" : "llama-3.1-8b-instant",
        "k": 5,
    },
)

View the evaluation results for experiment: 'test-hitler_llmops_dataset_2-qa-rag-a1a36558' at:
https://smith.langchain.com/o/36080d1a-1ea4-456c-a3cd-478d0d30e3a5/datasets/b2d9a30c-9aa7-42c1-b0eb-282584472869/compare?selectedSessions=0613b4a1-e455-4092-955f-b1070911969c




0it [00:00, ?it/s]{"timestamp": "2025-12-16T10:40:17.831485Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2025-12-16T10:40:17.832450Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2025-12-16T10:40:17.833067Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_6r...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2025-12-16T10:40:17.833447Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2025-12-16T10:40:17.835312Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20251216_154017_851dd74d", "temp_dir": "data\\session_20251216_154017_851dd74d", "faiss_dir": "faiss_index\\session_20251216_154017_851dd74d", "sessionized": true, "timestamp": "2025-12-16T10:40:17.836508Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "Hitler.txt", "saved_as": "